# Football Player Scouting & Recruitment Recommendation System

## 03 — Recommendation System

This notebook builds the first interpretable player recommendation system.

Given a recruitment requirement, the system:

1. Filters eligible players
2. Applies recruitment constraints
3. Scores players according to the target position
4. Allows custom priorities
5. Adds compatible playstyle bonuses
6. Produces explanations for the recommendations

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.features import (
    ABILITY_GROUPS,
    INDIVIDUAL_ABILITIES,
    normalize_attributes,
    create_ability_features,
    prepare_playstyles,
    ABILITY_FEATURES
)

from src.scoring import (
    POSITION_WEIGHTS,
    build_custom_weights,
    explain_player,
)

from src.recommendation import (
    recommend_players,
    get_position_candidates,
)

In [2]:
DATA_PATH = PROJECT_ROOT / "Data" / "players_cleaned.csv"

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["birthdate"]
)

df_normalized = normalize_attributes(df)
df_normalized = create_ability_features(df_normalized)
df_normalized = prepare_playstyles(df_normalized)

df_normalized["age"] = (
    2026
    - df_normalized["birthdate"].dt.year
    - (
        (
            df_normalized["birthdate"].dt.month > 9
        )
        |
        (
            (df_normalized["birthdate"].dt.month == 9)
            &
            (df_normalized["birthdate"].dt.day > 12)
        )
    ).astype(int)
)

In [3]:
from src.preprocessing import add_age

df_normalized = add_age(
    df_normalized,
    reference_date=pd.Timestamp("2026-09-12")
)

In [4]:
position = "CDM"
max_age = 24
gender = "Men's Football"

desired_playstyles = [
    "Intercept",
    "Anticipate",
    "Press Proven",
]

In [5]:
candidates = get_position_candidates(
    df_normalized,
    position
)

candidates = candidates[
    candidates["age"] <= max_age
].copy()

if gender is not None:
    candidates = candidates[
        candidates["gender"] == gender
    ]

print(f"Eligible candidates: {len(candidates)}")

Eligible candidates: 1861


In [6]:
from src.scoring import score_players

base_weights = POSITION_WEIGHTS[position]

candidates["profile_score"] = score_players(
    candidates,
    base_weights
)

In [7]:
candidates = candidates.sort_values(
    "profile_score",
    ascending=False
)

candidates[
    [
        "first_name",
        "last_name",
        "club",
        "position",
        "age",
        "profile_score"
    ]
].head(10)

,first_name,last_name,club,position,age,profile_score
43,João Pedro,Gonçalves Neves,Paris SG,CM,21,0.919055
10,Pedro,González López,FC Barcelona,CM,23,0.888649
95,Moisés,Caicedo,Chelsea,CDM,24,0.887236
144,Ryan,Gravenberch,Liverpool,CDM,24,0.866047
212,Elliot,Anderson,Manchester City,CDM,23,0.865983
314,Warren,Zaïre-Emery,Paris SG,CM,20,0.853264
321,Aleksandar,Pavlović,FC Bayern München,CDM,22,0.849383
535,Eduardo,Camavinga,Real Madrid,CM,23,0.847306
443,Victor,Froholdt,FC Porto,CM,20,0.842982
414,Adam,Wharton,Crystal Palace,CM,22,0.838539


In [8]:
cdm_priorities = {
    "defending": "very_high",
    "passing_technique": "high",
    "physical": "medium",
    "pace": "low",
}

custom_weights = build_custom_weights(
    base_weights,
    cdm_priorities
)

custom_weights

{'defending': 0.3731343283582089,
 'tackling': 0.14925373134328357,
 'passing_technique': 0.20149253731343283,
 'mentality_vision': 0.08955223880597014,
 'physical': 0.07462686567164178,
 'mentality_aggression': 0.05970149253731343,
 'dribbling_control': 0.05223880597014926}

In [9]:
candidates["custom_score"] = score_players(
    candidates,
    custom_weights
)

candidates.sort_values(
    "custom_score",
    ascending=False
)[
    [
        "first_name",
        "last_name",
        "club",
        "position",
        "age",
        "custom_score"
    ]
].head(10)

,first_name,last_name,club,position,age,custom_score
43,João Pedro,Gonçalves Neves,Paris SG,CM,21,0.922892
95,Moisés,Caicedo,Chelsea,CDM,24,0.894561
10,Pedro,González López,FC Barcelona,CM,23,0.889725
144,Ryan,Gravenberch,Liverpool,CDM,24,0.876098
212,Elliot,Anderson,Manchester City,CDM,23,0.868972
314,Warren,Zaïre-Emery,Paris SG,CM,20,0.858495
321,Aleksandar,Pavlović,FC Bayern München,CDM,22,0.855417
535,Eduardo,Camavinga,Real Madrid,CM,23,0.850425
443,Victor,Froholdt,FC Porto,CM,20,0.849408
414,Adam,Wharton,Crystal Palace,CM,22,0.844439


In [10]:
from src.recommendation import calculate_playstyle_score

In [11]:
PLAYSTYLE_BONUS_MAX = 0.05

candidates["playstyle_score"] = (
    candidates["playstyle_list"]
    .apply(
        lambda styles: calculate_playstyle_score(
            styles,
            desired_playstyles
        )
    )
)

candidates["playstyle_bonus"] = (
    candidates["playstyle_score"]
    * PLAYSTYLE_BONUS_MAX
)

candidates["final_score"] = (
    candidates["custom_score"]
    + candidates["playstyle_bonus"]
)

In [12]:
top_players = (
    candidates
    .sort_values("final_score", ascending=False)
    .head(10)
)

top_players[
    [
        "first_name",
        "last_name",
        "club",
        "position",
        "age",
        "custom_score",
        "playstyle_score",
        "playstyle_bonus",
        "final_score"
    ]
]

,first_name,last_name,club,position,age,custom_score,playstyle_score,playstyle_bonus,final_score
95,Moisés,Caicedo,Chelsea,CDM,24,0.894561,0.833333,0.041667,0.936228
43,João Pedro,Gonçalves Neves,Paris SG,CM,21,0.922892,0.250000,0.012500,0.935392
10,Pedro,González López,FC Barcelona,CM,23,0.889725,0.250000,0.012500,0.902225
144,Ryan,Gravenberch,Liverpool,CDM,24,0.876098,0.500000,0.025000,0.901098
212,Elliot,Anderson,Manchester City,CDM,23,0.868972,0.333333,0.016667,0.885639
443,Victor,Froholdt,FC Porto,CM,20,0.849408,0.500000,0.025000,0.874408
314,Warren,Zaïre-Emery,Paris SG,CM,20,0.858495,0.250000,0.012500,0.870995
321,Aleksandar,Pavlović,FC Bayern München,CDM,22,0.855417,0.250000,0.012500,0.867917
1623,Jobe,Bellingham,Borussia Dortmund,CM,20,0.839850,0.500000,0.025000,0.864850
414,Adam,Wharton,Crystal Palace,CM,22,0.844439,0.250000,0.012500,0.856939


In [13]:
player = top_players.iloc[0]

explanation = explain_player(
    player,
    custom_weights
)

explanation

,ability,player_value,weight,contribution
0,defending,0.936371,0.373134,0.349392
1,passing_technique,0.859810,0.201493,0.173245
2,tackling,0.934433,0.149254,0.139468
3,mentality_vision,0.800000,0.089552,0.071642
4,physical,0.833209,0.074627,0.062180
5,mentality_aggression,0.916667,0.059701,0.054726
6,dribbling_control,0.840523,0.052239,0.043908


In [14]:
explanation.head(5)

,ability,player_value,weight,contribution
0,defending,0.936371,0.373134,0.349392
1,passing_technique,0.859810,0.201493,0.173245
2,tackling,0.934433,0.149254,0.139468
3,mentality_vision,0.800000,0.089552,0.071642
4,physical,0.833209,0.074627,0.062180


In [15]:
from src.recommendation import get_playstyle_match

get_playstyle_match(
    player["playstyle_list"],
    desired_playstyles
)

2.5

## Conclusion

The recommendation system combines:

- position eligibility
- recruitment constraints
- position-specific ability weights
- customizable priorities
- playstyle compatibility
- interpretable player-level explanations

The system does not use EA's overall rating in the scoring process.

This makes the recommendation methodology transparent and configurable
rather than simply reproducing an existing aggregate rating.